# 📊 Trabalho de Clusterização em 3D com KMeans

## Disciplina: Aprendizado de Máquina Não Supervisionado  
**Curso:** Tecnologia em Ciência de Dados  
**Instituição:** Faculdade de Tecnologia e Inovação Senac DF  
**Professor:** Rogério Gomes Lopes  

**Objetivo:** Analisar a **Distribuição de Renda por Centis no Brasil** a partir de dados **granulares por Estado, Região e Ano**,  
por meio de **clusterização não supervisionada (KMeans)**, utilizando **3 variáveis contínuas** e visualização em **3D**.  
O propósito é identificar **padrões de renda, patrimônio e carga tributária** entre diferentes grupos socioeconômicos,  
explorando como esses perfis se distribuem entre estados e regiões, bem como sua evolução temporal.

---

### 👨‍🎓 Alunos
- Anderson de Matos Guimarães  
- Gustavo Stefano Thomazinho  
- Renan Ost  

---

📅 **Semestre:** 4º semestre (2025/2)  


## 1. Business Understanding

O presente trabalho busca analisar a **Distribuição de Renda por Centis no Brasil**, utilizando técnicas de 
**Aprendizado de Máquina Não Supervisionado**, com foco na **clusterização em 3D via KMeans**.

A base de dados disponibilizada pela Receita Federal apresenta informações **granulares** de renda declarada, 
discriminadas por:

- **Ano-calendário** (período de referência da declaração);
- **Ente Federativo (UF)**;
- **Centil de Renda** (100 divisões iguais da população, sendo que o centésimo é subdividido em extratos mais finos).

---

### 🔎 O que é um Centil?

Na base da Receita Federal, os declarantes são divididos em **100 grupos de igual tamanho**, chamados **centis**, 
a partir da **Renda Tributável Bruta (RTB)**:

- O **1º centil** corresponde ao 1% da população com menor renda declarada.  
- O **100º centil** corresponde ao 1% mais rico, que é subdividido em 10 partes iguais; e a última dessas (o 0,1% do topo) 
é novamente dividida em 10 partes, para discriminar com mais detalhe o **extrato superior da renda**.

Essa metodologia, baseada no conceito estatístico de **percentis**, permite analisar a distribuição de renda de forma 
**granular e comparável**, garantindo que cada centil represente a mesma quantidade de contribuintes.  
A subdivisão do centésimo centil é uma decisão metodológica da Receita Federal para melhor captar a 
**alta concentração de renda no topo**.

---

### Variáveis disponíveis na base
Para cada (Ano × UF × Centil), são informados valores como:

- **Rendimentos Tributáveis** (soma, limite, acumulada);  
- **Rendimentos Sujeitos à Tributação Exclusiva**;  
- **Rendimentos Isentos** (dividendos, Simples, outros);  
- **Despesas Dedutíveis** (saúde, educação, previdência, pensão, etc.);  
- **Imposto Devido**;  
- **Bens e Direitos** (imóveis, móveis, financeiros, outros);  
- **Dívidas e Ônus**.  

---

### 🎯 Objetivo da Análise
O objetivo é **identificar padrões de desigualdade econômica no Brasil**, por meio da clusterização de observações 
granulares (Centil × UF × Ano), utilizando **3 variáveis contínuas derivadas**:

1. **Renda Total** = Rendimentos Tributáveis + Exclusivos + Isentos  
2. **Patrimônio Líquido** = Bens Totais – Dívidas e Ônus  
3. **Carga Tributária Efetiva** = Imposto Devido / Renda Total  

---

### Questões de análise
- Quais perfis socioeconômicos são revelados pelos clusters?  
- Como esses clusters se distribuem entre **estados** e **regiões**?  
- Há diferenças significativas quando comparamos **anos distintos** (análise temporal)?  
- Os clusters ajudam a revelar **padrões de desigualdade de renda, patrimônio e carga tributária** no Brasil?


## 2. Data Understanding

Nesta etapa buscamos compreender a estrutura do dataset original, sua granularidade e o significado de suas variáveis.  
O objetivo é garantir que conhecemos bem os dados antes de realizar qualquer preparação ou modelagem.

---

### 📂 Dataset utilizado
A base **Distribuição de Renda por Centis** é disponibilizada pela Receita Federal.  
Cada linha corresponde a uma combinação de:

- **Ano-calendário** (período da declaração);  
- **Ente Federativo (UF)**;  
- **Centil de Renda** (100 divisões iguais da população com base na Renda Tributável Bruta).  

Ou seja, temos dados **granulares** por **ano, estado e centil**, o que permite uma análise detalhada da distribuição de renda no Brasil.

---

### 📖 Principais variáveis (segundo o dicionário de dados)
Algumas variáveis relevantes, extraídas do dicionário da Receita Federal:contentReference[oaicite:0]{index=0}, são:

- **Ano-calendário:** ano a que se refere a declaração.  
- **Ente Federativo:** estado (UF) ao qual os dados se referem.  
- **Centil:** grupo que representa 1% dos declarantes ordenados pela Renda Tributável Bruta.  
- **Rendimentos Tributáveis – Soma da RTB (R$ mi):** total da renda tributável bruta dentro do centil.  
- **Rendimentos Sujeitos à Tributação Exclusiva (R$ mi):** rendimentos sujeitos à tributação exclusiva e definitiva.  
- **Rendimentos Isentos (R$ mi):** incluem dividendos, Simples e outros rendimentos isentos.  
- **Despesas Dedutíveis (R$ mi):** previdência, dependentes, instrução, médicas, pensão, livro-caixa.  
- **Imposto Devido (R$ mi):** total de imposto calculado para o centil.  
- **Bens e Direitos (R$ mi):** imóveis, móveis, financeiros, outros.  
- **Dívidas e Ônus (R$ mi):** valor total de dívidas declaradas.  

Essas variáveis servirão de base para criarmos as **três variáveis derivadas** que serão utilizadas na clusterização.

---

### 🔎 Exploração inicial do dataset
Antes de criar variáveis derivadas, vamos observar o dataset original.


In [1]:
# ================================
# Carregar o dataset original
# ================================
import pandas as pd

# Como o arquivo está no mesmo diretório do notebook:
csv_path = "distribuicao-renda.csv"

# Em alguns casos o separador pode ser ";", por isso testamos
df = pd.read_csv(csv_path, sep=";", low_memory=False)


In [2]:
print("Dimensões do dataset:", df.shape)


Dimensões do dataset: (46350, 24)


In [3]:
# Primeiras linhas do dataset
df.head()


,Ano-calendário,Ente Federativo,Centil,Quantidade de Contribuintes,Rendimentos Tributaveis - Limite Superior da RTB do Centil [R$ milhões],Rendimentos Tributaveis - Soma da RTB do Centil [R$ milhões],Rendimentos Tributaveis - RTB Acumulada do Centil [R$ milhões],Rendimentos Tributaveis - Média da RTB do Centil [R$],Rendimentos Sujeitos à Tribut. Exclusiva [R$ milhões],Rendimentos Isentos - Lucros e dividendos [R$ milhões],...,Despesas Dedutíveis - Instrução [R$ milhões],Despesas Dedutíveis - Médicas [R$ milhões],Despesas Dedutíveis - Pensão Alimentícia [R$ milhões],Despesas Dedutíveis - Livro-Caixa [R$ milhões],Imposto Devido [R$ milhões],Bens e Direitos - Imóveis [R$ milhões],Bens e Direitos - Móveis [R$ milhões],Bens e Direitos - Financeiros [R$ milhões],Bens e Direitos - Outros Bens e Direitos [R$ milhões],Dívidas e Ônus [R$ milhões]
0,2006,BRASIL,1,241.563,NaN,NaN,NaN,NaN,"235,61","481,27",...,NaN,NaN,NaN,NaN,"0,16","5.281,59","686,21","6.549,15","1.006,40","1.610,39"
1,2006,BRASIL,2,241.563,NaN,NaN,NaN,NaN,"208,74","483,44",...,NaN,NaN,NaN,NaN,"0,22","5.295,48","668,82","5.762,77","681,75","694,12"
2,2006,BRASIL,3,241.562,NaN,NaN,NaN,NaN,"219,96","459,87",...,NaN,NaN,NaN,NaN,"0,31","5.566,27","670,64","5.451,95","377,17","650,98"
3,2006,BRASIL,4,241.563,NaN,NaN,NaN,NaN,"257,01","481,93",...,NaN,NaN,NaN,NaN,"0,17","5.860,02","678,44","6.104,09","256,16","1.079,20"
4,2006,BRASIL,5,241.562,NaN,NaN,NaN,NaN,"249,88","464,23",...,NaN,NaN,NaN,NaN,"0,17","5.193,31","682,38","5.592,52","269,28","671,97"


📌 **Análise do head():**  
- Confirma a granularidade dos dados: cada linha corresponde a **Ano × Ente Federativo × Centil**.  
- Observa-se que o campo **Ente Federativo = BRASIL** representa o nível agregado nacional, que deverá ser removido
nas análises, pois o foco é a comparação por estados (UF).  
- Colunas principais visíveis: rendimentos tributáveis, exclusivos, isentos, despesas dedutíveis, imposto devido, bens e direitos, dívidas.  
- Valores em **milhões de R$** (atenção na interpretação dos números).  
- Alguns campos aparecem como `NaN` no agregado “Brasil”, reforçando a necessidade de filtrar apenas os dados de UF.  


In [4]:
# Últimas linhas do dataset
df.tail()


,Ano-calendário,Ente Federativo,Centil,Quantidade de Contribuintes,Rendimentos Tributaveis - Limite Superior da RTB do Centil [R$ milhões],Rendimentos Tributaveis - Soma da RTB do Centil [R$ milhões],Rendimentos Tributaveis - RTB Acumulada do Centil [R$ milhões],Rendimentos Tributaveis - Média da RTB do Centil [R$],Rendimentos Sujeitos à Tribut. Exclusiva [R$ milhões],Rendimentos Isentos - Lucros e dividendos [R$ milhões],...,Despesas Dedutíveis - Instrução [R$ milhões],Despesas Dedutíveis - Médicas [R$ milhões],Despesas Dedutíveis - Pensão Alimentícia [R$ milhões],Despesas Dedutíveis - Livro-Caixa [R$ milhões],Imposto Devido [R$ milhões],Bens e Direitos - Imóveis [R$ milhões],Bens e Direitos - Móveis [R$ milhões],Bens e Direitos - Financeiros [R$ milhões],Bens e Direitos - Outros Bens e Direitos [R$ milhões],Dívidas e Ônus [R$ milhões]
46345,2020,TO,100.6,169.0,"486.602,57","79,96","434,49","473.152,32","7,78","2,64",...,"0,44","3,2","0,94","2,29","16,12","185,25","16,75","61,71","13,29","27,73"
46346,2020,TO,100.7,169.0,"523.830,78","85,51",520,"505.995,90","8,91","3,38",...,"0,43","3,4","1,21","2,46","17,67","195,46","22,77","80,08","3,55","43,45"
46347,2020,TO,100.8,169.0,"589.084,62","93,48","613,49","553.159,76","11,9","12,9",...,"0,52","3,11","1,01","3,35","19,68","457,81","28,19","169,15","25,91","43,73"
46348,2020,TO,100.9,169.0,"749.691,30","110,74","724,23","655.252,33","6,82","12,17",...,"0,39","2,25","1,58","8,62","23,19","182,83","28,11","122,92","26,32","26,01"
46349,2020,TO,100.10,168.0,"18.277.661,84","238,8","963,03","1.421.432,54","904,35","16,15",...,"0,28","2,21","1,06","58,95","45,22","514,07","31,17","1.137,50","219,8","190,09"


📌 **Análise do tail():**  
- Confirma que o dataset cobre o período até **2020**.  
- Mostra registros do estado de **Tocantins (TO)** no **100º centil** subdividido (100.1 até 100.10).  
- Essa subdivisão é feita apenas no topo da distribuição para detalhar melhor a concentração de renda no 1% mais rico.  
- Diferentemente do `head()`, aqui não aparecem linhas agregadas como "BRASIL".  
- Os valores estão preenchidos, mas com magnitudes muito elevadas (na casa de milhões), evidenciando a concentração de renda no topo.  
- Não há linhas adicionais de “Total” ou agregados no final, apenas dados regulares.  


In [5]:
# ================================
# Estatísticas descritivas
# ================================
df.describe().T


,count,mean,std,min,25%,50%,75%,max
Ano-calendário,46350.0,2013.000000,4.320540,2006.000,2009.000,2013.00,2017.000,2020.0
Quantidade de Contribuintes,46350.0,96.555265,197.999486,1.001,2.894,6.79,29.169,993.0


📌 **Análise do describe():**  
- O resumo estatístico (`describe()`) apresenta apenas duas variáveis: **Ano-calendário** e **Quantidade de Contribuintes**.  
- Isso acontece porque as demais colunas (valores de rendimentos, bens, dívidas etc.) foram carregadas como **texto (string)**,  
  já que utilizam **vírgula como separador decimal** e **ponto como separador de milhar**.  
- Portanto, o pandas não as reconhece inicialmente como numéricas, tratando-as como `object`.  
- Essa constatação é importante: mostra que será necessário realizar **ajustes e conversões de tipos de dados** na etapa de **Data Preparation**.  

➡️ Aqui, entretanto, nosso objetivo é apenas **entender o dataset original**:  
vemos que os anos variam de **2006 a 2020** e que a quantidade de contribuintes por centil está corretamente registrada,  
confirmando a granularidade da base.  


In [6]:
# Estrutura do dataset: tipos de dados e valores nulos
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46350 entries, 0 to 46349
Data columns (total 24 columns):
 #   Column                                                                       Non-Null Count  Dtype  
---  ------                                                                       --------------  -----  
 0   Ano-calendário                                                               46350 non-null  int64  
 1   Ente Federativo                                                              46350 non-null  object 
 2   Centil                                                                       46350 non-null  object 
 3   Quantidade de Contribuintes                                                  46350 non-null  float64
 4   Rendimentos Tributaveis - Limite Superior da RTB do Centil [R$ milhões]      42935 non-null  object 
 5   Rendimentos Tributaveis - Soma da RTB do Centil [R$ milhões]                 42935 non-null  object 
 6   Rendimentos Tributaveis - RTB Acumulad

📌 **Análise do info():**

- O dataset possui **46.350 registros** e **24 colunas**.  

### Tipos de dados
- **Numéricos:**
  - `Ano-calendário` → `int64`  
  - `Quantidade de Contribuintes` → `float64`
- **Categóricos (texto):**
  - `Ente Federativo` (UFs e o agregado "BRASIL")  
  - `Centil` (1 a 100, com subdivisões do 100º centil)  
- **Monetários (mas carregados como texto):**
  - 20 colunas, como *Rendimentos Tributáveis*, *Rendimentos Exclusivos*, *Rendimentos Isentos*, *Despesas Dedutíveis*, *Imposto Devido*, *Bens e Direitos*, *Dívidas e Ônus* → todas `object` (string), quando deveriam ser `float`.

### Valores ausentes (NaN)
- Diversas colunas monetárias têm menos de 46.350 registros válidos, o que indica **dados ausentes**:
  - `Imposto Devido` → 38.184 não nulos (~18% faltantes).  
  - `Livro-Caixa` → 39.593 não nulos (~15% faltantes).  
  - Outras colunas (dedutíveis, rendimentos tributáveis, etc.) também apresentam lacunas.  

### Problema principal
- O uso de **vírgula como separador decimal** e **ponto como separador de milhar** fez com que o pandas interpretasse 
as colunas monetárias como **texto (`object`)** em vez de numéricas.  
- Isso explica porque, no `describe()`, apenas duas colunas foram reconhecidas como numéricas.  

---

➡️ **Conclusão desta análise:**  
- O dataset bruto está coerente em estrutura, mas contém **problemas de tipos de dados** e **valores faltantes**.  
- Será necessário, na **Seção 3 (Data Preparation)**:
  1. Converter colunas monetárias para `float`.  
  2. Tratar valores ausentes.  
  3. Excluir agregados como "BRASIL".  
  4. Criar variáveis derivadas para a clusterização (Renda Total, Patrimônio Líquido, Carga Tributária).  


## 3. Data Preparation

Nesta etapa realizaremos a **preparação dos dados** para viabilizar a aplicação do algoritmo de **clusterização (KMeans)**.  
Como identificado na fase anterior (*Data Understanding*), o dataset apresenta:

- **Colunas monetárias carregadas como texto** (`object`), devido ao uso de vírgula como separador decimal e ponto como separador de milhar;  
- **Valores ausentes** em variáveis relevantes, como `Imposto Devido` e despesas dedutíveis;  
- **Linhas agregadas** para o nível "BRASIL", que não devem ser utilizadas na análise de estados (UF);  
- **Granularidade** baseada em (Ano × UF × Centil), que precisa ser preservada.

---

### 🎯 Objetivos da preparação
1. **Limpeza de dados:**
   - Remover linhas agregadas como "BRASIL".  
   - Tratar valores ausentes de forma adequada (remoção ou substituição).  

2. **Conversão de tipos:**
   - Transformar as colunas monetárias de `object` para `float`.  
   - Garantir que todos os valores estejam em formato numérico consistente (R$ milhões → `float`).  

3. **Criação das variáveis derivadas:**
   - **Renda Total** = Rendimentos Tributáveis + Exclusivos + Isentos  
   - **Patrimônio Líquido** = Bens Totais – Dívidas e Ônus  
   - **Carga Tributária Efetiva** = Imposto Devido / Renda Total  

4. **Seleção temporal:**
   - Para a análise inicial (fotografia), utilizar apenas o **ano mais recente** disponível.  
   - Posteriormente, explorar comparações temporais entre anos (evolução dos clusters).  

5. **Escalonamento:**
   - Aplicar **StandardScaler** para normalizar as variáveis contínuas, evitando distorções no KMeans.  

---

📌 Ao final desta etapa, teremos um **dataframe preparado para clusterização**, contendo apenas três variáveis contínuas 
(`Renda Total`, `Patrimônio Líquido`, `Carga Tributária`) mais as colunas de identificação (`Ano`, `UF`, `Centil`, `Região`).


In [7]:
# ================================
# Criação do dataframe de preparação
# ================================

# Mantemos o df original como dataset bruto
# Criamos uma cópia para começar a limpeza e preparação
df_prep = df.copy()

print("Dimensões do dataframe bruto:", df.shape)
print("Dimensões do dataframe para preparação:", df_prep.shape)


Dimensões do dataframe bruto: (46350, 24)
Dimensões do dataframe para preparação: (46350, 24)


📌 **Início da preparação dos dados:**

- Mantemos o `df` como **dataset bruto**, carregado diretamente do CSV.  
- Criamos uma cópia chamada `df_prep`, que será utilizada para **todas as etapas de preparação**:  
  - limpeza de linhas agregadas (ex.: "BRASIL");  
  - conversão das colunas monetárias para formato numérico (`float`);  
  - tratamento de valores ausentes;  
  - criação das variáveis derivadas (`Renda Total`, `Patrimônio Líquido`, `Carga Tributária`).  

➡️ Essa separação garante que possamos **comparar o dataset original com o preparado**, tornando o processo mais didático e reprodutível.


In [8]:
# ================================
# Remover linhas agregadas do Brasil
# ================================

print("Registros antes da limpeza:", df_prep.shape[0])

# Normaliza o texto para evitar diferenças de caixa/letras
df_prep = df_prep[df_prep["Ente Federativo"].str.upper() != "BRASIL"]

print("Registros após a limpeza:", df_prep.shape[0])
print("Entes federativos únicos:", df_prep["Ente Federativo"].nunique())
df_prep["Ente Federativo"].unique()


Registros antes da limpeza: 46350
Registros após a limpeza: 44550
Entes federativos únicos: 27


array(['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG',
       'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR',
       'RS', 'SC', 'SE', 'SP', 'TO'], dtype=object)

📌 **Resultados da limpeza do agregado 'BRASIL':**

- O dataset original possuía **46.350 registros**.  
- Após a remoção das linhas com `Ente Federativo = BRASIL`, restaram **44.550 registros**.  
- O número de entes federativos únicos agora é **27**, correspondendo corretamente aos **26 estados brasileiros + Distrito Federal**.  
- A lista de UFs confirma a abrangência nacional:  
  `['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG',
   'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR',
   'RS', 'SC', 'SE', 'SP', 'TO']`

➡️ Com essa limpeza, garantimos que a análise seja feita apenas no nível estadual, mantendo a granularidade desejada para os **27 entes federativos**.


In [10]:
# ================================
# Conversão das colunas monetárias para float (com segurança)
# ================================

def to_float(series):
    return pd.to_numeric(
        series.astype(str)
              .str.replace(".", "", regex=False)   # remove separador de milhar
              .str.replace(",", ".", regex=False), # troca vírgula decimal por ponto
        errors="coerce"  # valores inválidos viram NaN
    )

# Aplicar a conversão
for col in cols_monetarias:
    if col in df_prep.columns:
        df_prep[col] = to_float(df_prep[col])

# Conferir novamente
df_prep.info()


<class 'pandas.core.frame.DataFrame'>
Index: 44550 entries, 120 to 46349
Data columns (total 24 columns):
 #   Column                                                                       Non-Null Count  Dtype  
---  ------                                                                       --------------  -----  
 0   Ano-calendário                                                               44550 non-null  int64  
 1   Ente Federativo                                                              44550 non-null  object 
 2   Centil                                                                       44550 non-null  object 
 3   Quantidade de Contribuintes                                                  44550 non-null  float64
 4   Rendimentos Tributaveis - Limite Superior da RTB do Centil [R$ milhões]      41250 non-null  float64
 5   Rendimentos Tributaveis - Soma da RTB do Centil [R$ milhões]                 41250 non-null  float64
 6   Rendimentos Tributaveis - RTB Acumulada d

📌 **Resultados da conversão numérica:**

- Todas as **20 colunas monetárias** foram convertidas para o tipo `float64`.  
- A estrutura atual do dataset é:  
  - `Ano-calendário` → `int64`  
  - `Quantidade de Contribuintes` + todas as variáveis monetárias → `float64`  
  - `Ente Federativo` e `Centil` → `object`  

- **Valores ausentes (NaN):**  
  - Permanecem em várias colunas, como esperado.  
  - Exemplos: `Imposto Devido` (~19% ausente), `Livro-Caixa` (~15% ausente).  
  - Esses casos serão tratados posteriormente, quando definirmos quais variáveis entram na análise.  

➡️ Agora o dataset está **numericamente padronizado**, pronto para a criação das variáveis derivadas necessárias à clusterização.


In [11]:
# ================================
# Remover linhas com NaN nas variáveis necessárias
# ================================

# Colunas que precisamos para criar as variáveis derivadas
colunas_chave = [
    'Rendimentos Tributaveis - Soma da RTB do Centil [R$ milhões]',
    'Rendimentos Sujeitos à Tribut. Exclusiva [R$ milhões]',
    'Rendimentos Isentos - Lucros e dividendos [R$ milhões]',
    'Rendimentos Isentos - Rendim. Sócio/Titular ME/EPP Opt SIMPLES [R$ milhões]',
    'Rendimentos Isentos - Outros Rendimentos Isentos [R$ milhões]',
    'Bens e Direitos - Imóveis [R$ milhões]',
    'Bens e Direitos - Móveis [R$ milhões]',
    'Bens e Direitos - Financeiros [R$ milhões]',
    'Bens e Direitos - Outros Bens e Direitos [R$ milhões]',
    'Dívidas e Ônus [R$ milhões]',
    'Imposto Devido [R$ milhões]'
]

print("Registros antes da remoção:", df_prep.shape[0])

# Remover linhas com valores ausentes apenas nas colunas-chave
df_prep = df_prep.dropna(subset=colunas_chave)

print("Registros após a remoção:", df_prep.shape[0])


Registros antes da remoção: 44550
Registros após a remoção: 34604


📌 **Resultados da remoção de valores ausentes:**

- Registros antes da remoção: **44.550**  
- Registros após a remoção: **34.604**  
- Foram eliminados **9.946 registros (~22%)** que continham valores ausentes em pelo menos uma das colunas-chave.  

➡️ Agora o `df_prep` contém apenas observações **completas** para a criação das variáveis derivadas.  
Essa escolha reduz a amostra, mas aumenta a **qualidade e consistência dos dados**, garantindo que os clusters reflitam informações confiáveis.


In [12]:
# ================================
# Criação das variáveis derivadas
# ================================

# Renda Total = Tributáveis + Exclusivos + Isentos (3 colunas de isentos)
df_prep["Renda Total"] = (
    df_prep['Rendimentos Tributaveis - Soma da RTB do Centil [R$ milhões]'] +
    df_prep['Rendimentos Sujeitos à Tribut. Exclusiva [R$ milhões]'] +
    df_prep['Rendimentos Isentos - Lucros e dividendos [R$ milhões]'] +
    df_prep['Rendimentos Isentos - Rendim. Sócio/Titular ME/EPP Opt SIMPLES [R$ milhões]'] +
    df_prep['Rendimentos Isentos - Outros Rendimentos Isentos [R$ milhões]']
)

# Patrimônio Líquido = Bens (4 colunas) – Dívidas
df_prep["Patrimônio Líquido"] = (
    df_prep['Bens e Direitos - Imóveis [R$ milhões]'] +
    df_prep['Bens e Direitos - Móveis [R$ milhões]'] +
    df_prep['Bens e Direitos - Financeiros [R$ milhões]'] +
    df_prep['Bens e Direitos - Outros Bens e Direitos [R$ milhões]'] -
    df_prep['Dívidas e Ônus [R$ milhões]']
)

# Carga Tributária Efetiva = Imposto Devido / Renda Total
df_prep["Carga Tributária Efetiva"] = (
    df_prep['Imposto Devido [R$ milhões]'] / df_prep["Renda Total"]
)

# Conferir as novas colunas
df_prep[["Renda Total", "Patrimônio Líquido", "Carga Tributária Efetiva"]].head()


,Renda Total,Patrimônio Líquido,Carga Tributária Efetiva
120,1.33,14.16,0.0
121,1.03,1.50,0.0
122,1.05,2.97,0.0
123,1.52,3.03,0.0
124,0.95,1.96,0.0


📌 **Criação das variáveis derivadas:**

Foram geradas três novas variáveis a partir das colunas monetárias:

1. **Renda Total**  
   - Soma de todos os rendimentos:  
     - Rendimentos Tributáveis  
     - Rendimentos Exclusivos  
     - Rendimentos Isentos (dividendos, Simples, outros)  
   - Exemplo: linha 120 apresenta `Renda Total = 1.33 (milhões de R$)`.

2. **Patrimônio Líquido**  
   - Diferença entre o total de bens e direitos (imóveis, móveis, financeiros, outros) e as dívidas/ônus.  
   - Exemplo: linha 120 apresenta `Patrimônio Líquido = 14.16 (milhões de R$)`.

3. **Carga Tributária Efetiva**  
   - Razão entre o Imposto Devido e a Renda Total.  
   - Observamos que, nas primeiras linhas, o valor é **0.0**, indicando que nesses centis o imposto devido foi nulo 
   (mesmo havendo renda declarada).

---

➡️ Essas três variáveis sintetizam os principais aspectos da distribuição de renda:  
- **fluxo de renda** (Renda Total),  
- **estoque de riqueza** (Patrimônio Líquido),  
- **peso dos tributos** (Carga Tributária Efetiva).  

Elas serão a base para a clusterização em 3D com o algoritmo **KMeans**.


In [14]:
df_prep

,Ano-calendário,Ente Federativo,Centil,Quantidade de Contribuintes,Rendimentos Tributaveis - Limite Superior da RTB do Centil [R$ milhões],Rendimentos Tributaveis - Soma da RTB do Centil [R$ milhões],Rendimentos Tributaveis - RTB Acumulada do Centil [R$ milhões],Rendimentos Tributaveis - Média da RTB do Centil [R$],Rendimentos Sujeitos à Tribut. Exclusiva [R$ milhões],Rendimentos Isentos - Lucros e dividendos [R$ milhões],...,Despesas Dedutíveis - Livro-Caixa [R$ milhões],Imposto Devido [R$ milhões],Bens e Direitos - Imóveis [R$ milhões],Bens e Direitos - Móveis [R$ milhões],Bens e Direitos - Financeiros [R$ milhões],Bens e Direitos - Outros Bens e Direitos [R$ milhões],Dívidas e Ônus [R$ milhões],Renda Total,Patrimônio Líquido,Carga Tributária Efetiva
120,2006,AC,1,251.0,0.00,0.00,0.00,0.00,0.00,0.33,...,0.00,0.00,12.29,0.30,1.48,0.09,0.00,1.33,14.16,0.000000
121,2006,AC,2,250.0,0.00,0.00,0.00,0.00,0.01,0.03,...,0.00,0.00,1.43,0.40,1.25,0.25,1.83,1.03,1.50,0.000000
122,2006,AC,3,250.0,0.00,0.00,0.00,0.00,0.01,0.02,...,0.00,0.00,1.83,0.28,0.82,0.04,0.00,1.05,2.97,0.000000
123,2006,AC,4,250.0,0.00,0.00,0.00,0.00,0.00,0.37,...,0.00,0.00,1.07,0.40,1.77,0.00,0.21,1.52,3.03,0.000000
124,2006,AC,5,251.0,0.00,0.00,0.00,0.00,0.05,0.10,...,0.00,0.00,1.22,0.25,0.78,0.05,0.34,0.95,1.96,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46345,2020,TO,100.6,169.0,486602.57,79.96,434.49,473152.32,7.78,2.64,...,2.29,16.12,185.25,16.75,61.71,13.29,27.73,124.66,249.27,0.129312
46346,2020,TO,100.7,169.0,523830.78,85.51,520.00,505995.90,8.91,3.38,...,2.46,17.67,195.46,22.77,80.08,3.55,43.45,118.46,258.41,0.149164
46347,2020,TO,100.8,169.0,589084.62,93.48,613.49,553159.76,11.90,12.90,...,3.35,19.68,457.81,28.19,169.15,25.91,43.73,130.66,637.33,0.150620
46348,2020,TO,100.9,169.0,749691.30,110.74,724.23,655252.33,6.82,12.17,...,8.62,23.19,182.83,28.11,122.92,26.32,26.01,153.20,334.17,0.151371


In [13]:
# ================================
# Estatísticas das variáveis derivadas
# ================================

df_prep[["Renda Total", "Patrimônio Líquido", "Carga Tributária Efetiva"]].describe().T


,count,mean,std,min,25%,50%,75%,max
Renda Total,34604.0,954.891539,3335.449283,0.29,106.477500,268.960000,767.877500,166267.660000
Patrimônio Líquido,34604.0,2498.691185,12444.749711,-87284.07,137.935000,437.570000,1640.117500,720926.210000
Carga Tributária Efetiva,34604.0,0.040340,0.050576,0.00,0.000456,0.014868,0.067872,0.207894
